# 🎙️ Kokoro TTS Studio

Text-to-Speech with Single & Multi Speaker modes

In [ ]:
%pip install gradio pydub numpy -q

In [ ]:
%pip install kokoro TTS deep-translator -q

In [ ]:
import gradio as gr
from multispeaker import process_multispeaker_script, get_voice_names

VOICE_OPTIONS = {
    "American Female": ["af_heart", "af_bella", "af_nicole", "af_aoede", "af_sky", "af_sarah", "af_nova", "af_river"],
    "American Male": ["am_adam", "am_michael", "am_echo", "am_eric", "am_liam", "am_onyx"],
    "British Female": ["bf_emma", "bf_isabella", "bf_alice", "bf_lily"],
    "British Male": ["bm_george", "bm_lewis", "bm_daniel", "bm_fable"]
}
ALL_VOICES = [v for voices in VOICE_OPTIONS.values() for v in voices]
MAX_SPEAKERS = 10

In [ ]:
def build_speaker_data(text, current):
    if not text.strip():
        return [{"name": f"Speaker {i+1}", "voice": ALL_VOICES[i % len(ALL_VOICES)]} for i in range(2)]
    speakers, seen = [], set()
    for line in text.strip().split('\n'):
        if ':' in line:
            speaker = line.split(':', 1)[0].strip()
            if speaker not in seen and speaker:
                seen.add(speaker)
                voice = next((s["voice"] for s in current if s["name"] == speaker), "af_heart")
                speakers.append({"name": speaker, "voice": voice})
    while len(speakers) < 2:
        num = len(speakers) + 1
        speakers.append({"name": f"Speaker {num}", "voice": ALL_VOICES[(num-1) % len(ALL_VOICES)]})
    return speakers[:MAX_SPEAKERS]

def update_ui(text, data):
    new_data = build_speaker_data(text, data)
    updates = []
    for i in range(MAX_SPEAKERS):
        if i < len(new_data):
            updates.extend([gr.update(value=new_data[i]["name"], visible=True), gr.update(value=new_data[i]["voice"], visible=True), gr.update(visible=True)])
        else:
            updates.extend([gr.update(value="", visible=False), gr.update(value="af_heart", visible=False), gr.update(visible=False)])
    return new_data, updates

def add_speaker(data):
    if len(data) < MAX_SPEAKERS:
        data.append({"name": f"Speaker {len(data)+1}", "voice": "af_heart"})
    return data, build_updates(data)

def remove_speaker(data, idx):
    if 0 <= idx < len(data): data.pop(idx)
    return data, build_updates(data)

def build_updates(data):
    updates = []
    for i in range(MAX_SPEAKERS):
        if i < len(data):
            updates.extend([gr.update(value=data[i]["name"], visible=True), gr.update(value=data[i]["voice"], visible=True), gr.update(visible=True)])
        else:
            updates.extend([gr.update(value="", visible=False), gr.update(value="af_heart", visible=False), gr.update(visible=False)])
    return updates

def update_name(data, idx, name):
    if 0 <= idx < len(data) and name.strip(): data[idx]["name"] = name.strip()
    return data

def update_voice(data, idx, voice):
    if 0 <= idx < len(data): data[idx]["voice"] = voice
    return data

def to_config(data):
    return '\n'.join([f"{s['name']}:{s['voice']}" for s in data])

def generate_single(text, voice, speaker):
    if not text.strip(): return None, "Please enter text"
    try:
        config = f"{speaker or 'Speaker'}:{voice}"
        script = f"{speaker or 'Speaker'}: {text}"
        output, _, info = process_multispeaker_script(script, config, "American English", 1.0, 0.2, True, False)
        return output, f"✅ Generated with {voice}"
    except Exception as e: return None, f"❌ Error: {str(e)}"

def generate_multi(text, data):
    if not text.strip(): return None, "Please enter script text"
    try:
        config = to_config(data)
        output, _, info = process_multispeaker_script(text, config, "American English", 1.0, 0.2, True, False)
        return output, info
    except Exception as e: return None, f"❌ Error: {str(e)}"

In [ ]:
with gr.Blocks(
    theme=gr.themes.Base(primary_hue="blue", secondary_hue="slate", neutral_hue="slate").set(
        body_background_fill="#1a1a2e", block_background_fill="#16213e",
        block_label_background_fill="#0f3460", block_title_background_fill="#0f3460",
        input_background_fill="#1a1a2e", button_primary_background_fill="#e94560",
        button_primary_text_color="white", button_secondary_background_fill="#0f3460",
        body_text_color="#eaeaea"
    ),
    title="Kokoro TTS Studio",
    css=".gradio-container{max-width:1400px!important}.speaker-card{border:1px solid #0f3460;border-radius:8px;padding:12px;margin-bottom:12px}.generate-btn{font-size:16px;font-weight:bold;padding:12px 32px}"
) as demo:
    
    gr.Markdown("# 🎙️ Kokoro TTS Studio\nText-to-Speech with multiple voices")
    
    with gr.Tabs():
        # Single Speaker
        with gr.TabItem("🎤 Single Speaker"):
            with gr.Row():
                with gr.Column(scale=2):
                    single_text = gr.Textbox(label="Text", placeholder="Enter text...", lines=8, show_copy_button=True)
                    with gr.Row():
                        single_voice = gr.Dropdown(label="Voice", choices=ALL_VOICES, value="af_heart")
                        single_speaker = gr.Textbox(label="Speaker Name", placeholder="Speaker 1")
                    single_generate = gr.Button("🎵 Generate Audio", variant="primary", size="lg", elem_classes="generate-btn")
                with gr.Column(scale=1):
                    gr.Markdown("### 🔊 Preview")
                    single_audio = gr.Audio(label="Generated Audio", type="filepath", show_download_button=True)
                    single_info = gr.Markdown("")
            single_generate.click(fn=generate_single, inputs=[single_text, single_voice, single_speaker], outputs=[single_audio, single_info])
        
        # Multi Speaker
        with gr.TabItem("👥 Multi Speaker"):
            speaker_state = gr.State([{"name": "Speaker 1", "voice": "af_bella"}, {"name": "Speaker 2", "voice": "bf_isabella"}])
            with gr.Row():
                with gr.Column(scale=1):
                    gr.Markdown("### 📝 Script")
                    multi_script = gr.Textbox(label="Script", placeholder="Speaker 1: Hello\nSpeaker 2: Hi there!", lines=15, show_copy_button=True)
                with gr.Column(scale=1):
                    gr.Markdown("### 🎛️ Voice Settings")
                    cards = []
                    for i in range(MAX_SPEAKERS):
                        with gr.Group(visible=(i<2)):
                            with gr.Row(variant="panel", elem_classes="speaker-card"):
                                name_inp = gr.Textbox(label="Name", value=f"Speaker {i+1}" if i<2 else "", visible=(i<2))
                                voice_drop = gr.Dropdown(label="Voice", choices=ALL_VOICES, value="af_heart", visible=(i<2))
                                rem_btn = gr.Button("🗑️", variant="secondary", size="sm", visible=(i<2))
                        cards.append({"name": name_inp, "voice": voice_drop, "remove": rem_btn})
                    add_btn = gr.Button("➕ Add Speaker", variant="secondary", size="sm")
            with gr.Row():
                multi_generate = gr.Button("🎵 Generate Audio", variant="primary", size="lg", elem_classes="generate-btn")
            with gr.Row():
                multi_audio = gr.Audio(label="Generated Audio", type="filepath", show_download_button=True)
                multi_info = gr.Markdown("")
            
            multi_script.change(fn=update_ui, inputs=[multi_script, speaker_state], outputs=[speaker_state]+[c for i in range(MAX_SPEAKERS) for c in [cards[i]["name"],cards[i]["voice"],cards[i]["remove"]]])
            add_btn.click(fn=add_speaker, inputs=[speaker_state], outputs=[speaker_state]+[c for i in range(MAX_SPEAKERS) for c in [cards[i]["name"],cards[i]["voice"],cards[i]["remove"]]])
            for i in range(MAX_SPEAKERS):
                cards[i]["name"].change(fn=update_name, inputs=[speaker_state, gr.Number(value=i, visible=False), cards[i]["name"]], outputs=[speaker_state])
                cards[i]["voice"].change(fn=update_voice, inputs=[speaker_state, gr.Number(value=i, visible=False), cards[i]["voice"]], outputs=[speaker_state])
                cards[i]["remove"].click(fn=lambda d,idx=i: remove_speaker(d,idx), inputs=[speaker_state], outputs=[speaker_state]+[c for j in range(MAX_SPEAKERS) for c in [cards[j]["name"],cards[j]["voice"],cards[j]["remove"]]])
            multi_generate.click(fn=generate_multi, inputs=[multi_script, speaker_state], outputs=[multi_audio, multi_info])
    
    gr.Markdown("---\n**Kokoro TTS Studio** | Built with Gradio")

In [ ]:
demo.launch()